# Calculate the metrics from the csv

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from evaluation_utils import bootstrap_from_csv
import pickle
from pathlib import Path

# --- CALCULATION ---
# Updated to match the folder structure (with "evaluation/" prefix and without "_v1")
models_to_test = {
    "DINOv2 (Video)": "csvs/dinov2_open/open_dist_matrix.csv",
    "DINOv2 (Image)": "csvs/dinov2_open_image/open_dist_matrix.csv",
    "SwinV2 (Video)": "csvs/swin_open/open_dist_matrix.csv",
    "SwinV2 (Image)": "csvs/swin_open_image/open_dist_matrix.csv",
    "ViT (Video)":    "csvs/vit_open/open_dist_matrix.csv",
    "ViT (Image)":    "csvs/vit_open_image/open_dist_matrix.csv"
}

# Dictionary to store all computed metrics
all_open_results = {}

for name, path in models_to_test.items():
    print(f"\n{'='*30}\nProcessing {name}\n{'='*30}")
    # Increased 'm' to 100 for statistically sound confidence intervals
    all_open_results[name] = bootstrap_from_csv(path, m=100, mode="open")


# --- Save Configuration ---
SAVE_DIR = Path("results_data")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Define a filename (e.g., "closed_world_results.pkl")
FILENAME = f"open_world_benchmarks.pkl"
SAVE_PATH = SAVE_DIR / FILENAME

print(f"\n-> Saving all results to {SAVE_PATH}...")

with open(SAVE_PATH, 'wb') as f:
    pickle.dump(all_open_results, f)

print("✅ Results saved successfully.")

In [ ]:
# Load the saved data
LOAD_PATH = "results_data/open_world_benchmarks.pkl"

with open(LOAD_PATH, 'rb') as f:
    all_results = pickle.load(f)

print(f"Loaded results for: {list(all_results.keys())}")
# -------------------------------------------------------------
# --- Plotting Configuration ---
# -------------------------------------------------------------
MAX_FAR = 0.15    # Choose how far the X-axis goes (0.20 = 20% FAR)
FAR_TICK_STEP = 0.02  # How frequent the percentage ticks are (0.02 = every 2%)

# -------------------------------------------------------------
# --- Plotting Open-World Curves with Confidence Intervals ---
# -------------------------------------------------------------

plt.figure(figsize=(12, 7))
colors = plt.cm.get_cmap('tab10', len(all_open_results))

for i, (name, res) in enumerate(all_open_results.items()):
    fars = res["mean_fars"]
    dirs = res["mean_dirs"]
    dirs_lower = res["lower_dirs"]
    dirs_upper = res["upper_dirs"]
    color = colors(i)
    
    # 1. Plot the Shaded Uncertainty Interval (95% CI)
    plt.fill_between(fars, dirs_lower, dirs_upper, color=color, alpha=0.15)
    
    # 2. Plot the Mean Line
    # Solid for Video, Dashed for Image (if labeled)
    line_style = '-' if "(Video)" in name or "(Image)" not in name else '--'
    plt.plot(fars, dirs, color=color, linestyle=line_style, linewidth=2.5, label=name)

# Format the plot
plt.title(f"Open World: DIR vs FAR (up to {MAX_FAR:.0%} FAR)", fontsize=14, fontweight='bold')
plt.xlabel("False Accept Rate (FAR)", fontsize=12)
plt.ylabel("Detection and Identification Rate (DIR)", fontsize=12)

# --- Dynamic Axis Control ---
plt.xlim(0, MAX_FAR) 
plt.ylim(0, 0.35)
plt.xticks(np.arange(0, MAX_FAR + 0.01, step=FAR_TICK_STEP))

# Format the X-axis as percentages
ax = plt.gca()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=10, loc='lower right', frameon=True)
plt.tight_layout()

# -------------------------------------------------------------
# --- Print Final Metrics with Confidence Intervals ---
# -------------------------------------------------------------
print("\n" + "="*75)
print(f"{'Model & Modality':<25} | {'FAR Target':<12} | {'DIR (95% CI)':<25}")
print("="*75)

for name, res in all_open_results.items():
    fars = res["mean_fars"]
    
    for target in [0.01, 0.05, 0.10]:
        idx = np.argmin(np.abs(fars - target))
        d_mean = res["mean_dirs"][idx]
        d_low = res["lower_dirs"][idx]
        d_high = res["upper_dirs"][idx]
        
        # Only print the first row with the name for clarity
        row_name = name if target == 0.01 else ""
        print(f"{row_name:<25} | {target:>10.0%} | {d_mean:.2%} [{d_low:.2%} - {d_high:.2%}]")
    
    print("-" * 75)

plt.show()